In [18]:
import os
import pandas as pd
import numpy as np

In [19]:
SILVER_PATH = r"data/silver"
GOLD_PATH = r"data/gold"

os.makedirs(GOLD_PATH, exist_ok=True)


In [20]:
fin_txn_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\silver\finance_transactions_clean.csv")
user_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\silver\user_db_clean.csv")
merchant_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\silver\merchant_db_clean.csv")

In [21]:
fraud_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\silver\fraud_patterns_clean.csv")

In [22]:
def apply_amount_rule(amount, rule):
    """Check if transaction amount satisfies fraud rule"""
    if pd.isna(rule):
        return 0
    if '>' in rule:
        return int(amount > float(rule.replace('>', '')))
    elif '<' in rule:
        return int(amount < float(rule.replace('<', '')))
    return 0

In [23]:
df = fin_txn_df.merge(merchant_df, on='merchant_name', how='left')
df = df.merge(fraud_df, left_on='merchant_name', right_on='merchant', how='left')

df['rule_based_flag'] = df.apply(
    lambda row: apply_amount_rule(row['amount'], row['amount_range']),
    axis=1
)

df['is_high_value'] = (df['amount'] > 15000).astype(int)

In [24]:
df

,transaction_id,user_id,date,amount,merchant_name,payment_mode,category,user_category_new,clean_brand,is_refund,...,expected_category,avg_transaction_value,risk_level,risk_level_encoded,pattern_id,merchant,amount_range,fraud_flag,rule_based_flag,is_high_value
0,TXN0001,U038,2024-03-11,6614.32,Starbucks,UPI,other,Low,Starbucks,0,...,Food & Beverage,300,low,1,NaN,NaN,NaN,NaN,0,0
1,TXN0002,U033,2024-01-31,14567.58,Amazon,Cash,refund,Medium-Low,Amazon,1,...,Shopping,1500,medium,2,P002,Amazon,<50,0.0,0,0
2,TXN0003,U013,2024-01-13,12937.22,Local Shop,UPI,shopping,Medium-Low,Local Shop,0,...,Others,100,high,3,NaN,NaN,NaN,NaN,0,0
3,TXN0003,U013,2024-01-13,12937.22,Local Shop,UPI,shopping,Medium-Low,Local Shop,0,...,Others,500,high,3,NaN,NaN,NaN,NaN,0,0
4,TXN0004,U026,2024-02-13,5755.79,Starbucks,Card,shopping,Low,Starbucks,0,...,Food & Beverage,300,low,1,NaN,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1207,TXN0998,U029,2024-01-06,11386.02,Local Shop,Card,NaN,Medium-Low,Local Shop,0,...,Others,100,high,3,NaN,NaN,NaN,NaN,0,0
1208,TXN0998,U029,2024-01-06,11386.02,Local Shop,Card,NaN,Medium-Low,Local Shop,0,...,Others,500,high,3,NaN,NaN,NaN,NaN,0,0
1209,TXN0999,U002,2024-03-18,27224.41,DMart,Card,Food & Beverage,High,DMart,0,...,Grocery,1000,low,1,NaN,NaN,NaN,NaN,0,1
1210,TXN1000,U047,2024-01-08,11649.44,Local Shop,Card,electronics,Medium-Low,Local Shop,0,...,Others,100,high,3,NaN,NaN,NaN,NaN,0,0


In [25]:

merchant_agg = df.groupby('merchant_name').agg(

    # Risk Signals
    fraud_transaction_count=('fraud_flag', 'sum'),
    high_value_txn_count=('is_high_value', 'sum'),
    avg_transaction_amount=('amount', 'mean'),

    # Pattern Matching
    rule_based_hits=('rule_based_flag', 'sum'),

    # Volume
    total_txn=('transaction_id', 'count')

).reset_index()

In [26]:
merchant_agg

,merchant_name,fraud_transaction_count,high_value_txn_count,avg_transaction_amount,rule_based_hits,total_txn
0,Amazon,0.0,56,15838.715962,0,104
1,Apple,0.0,54,13980.855304,0,115
2,Big Bazaar,98.0,55,15726.970102,83,98
3,DMart,0.0,48,14675.352708,0,96
4,Flipkart,100.0,53,15063.526000,69,100
5,Local Shop,0.0,178,13508.563962,0,424
6,OLA,0.0,48,14725.376122,96,98
7,Starbucks,0.0,49,15988.024831,0,89
8,Swiggy,0.0,44,14728.739659,0,88


In [27]:
# Normalize signals
merchant_agg['fraud_ratio'] = merchant_agg['fraud_transaction_count'] / merchant_agg['total_txn']
merchant_agg['high_value_ratio'] = merchant_agg['high_value_txn_count'] / merchant_agg['total_txn']
merchant_agg['rule_hit_ratio'] = merchant_agg['rule_based_hits'] / merchant_agg['total_txn']

# weighted scoring 
merchant_agg['merchant_risk_score'] = (
    0.5 * merchant_agg['fraud_ratio'] +
    0.3 * merchant_agg['rule_hit_ratio'] +
    0.2 * merchant_agg['high_value_ratio']
)

merchant_agg = merchant_agg.merge(
    merchant_df[['merchant_name', 'risk_level_encoded', 'expected_category']],
    on='merchant_name',
    how='left'
)

In [28]:
merchant_agg

,merchant_name,fraud_transaction_count,high_value_txn_count,avg_transaction_amount,rule_based_hits,total_txn,fraud_ratio,high_value_ratio,rule_hit_ratio,merchant_risk_score,risk_level_encoded,expected_category
0,Amazon,0.0,56,15838.715962,0,104,0.0,0.538462,0.000000,0.107692,2,Shopping
1,Apple,0.0,54,13980.855304,0,115,0.0,0.469565,0.000000,0.093913,2,Electronics
2,Big Bazaar,98.0,55,15726.970102,83,98,1.0,0.561224,0.846939,0.866327,1,Grocery
3,DMart,0.0,48,14675.352708,0,96,0.0,0.500000,0.000000,0.100000,1,Grocery
4,Flipkart,100.0,53,15063.526000,69,100,1.0,0.530000,0.690000,0.813000,2,E-commerce
5,Local Shop,0.0,178,13508.563962,0,424,0.0,0.419811,0.000000,0.083962,3,Others
6,Local Shop,0.0,178,13508.563962,0,424,0.0,0.419811,0.000000,0.083962,3,Others
7,OLA,0.0,48,14725.376122,96,98,0.0,0.489796,0.979592,0.391837,1,Transport
8,Starbucks,0.0,49,15988.024831,0,89,0.0,0.550562,0.000000,0.110112,1,Food & Beverage
9,Swiggy,0.0,44,14728.739659,0,88,0.0,0.500000,0.000000,0.100000,1,Food & Beverage
